# 작은 diffusion LM으로 영어 동화 끝까지 생성하기

이 장에서 바뀌는 단 한 가지는 **생성 샘플러**입니다. Ch 32에서 같은 작은 diffusion LM(vocab 2048, 30000 step)이 이미 영어 동화를 생성했지만, 기본 샘플러(confidence remasking)는 한번 높은 확신을 받은 흔한 토큰을 거듭 뽑아 *반복*이 보였습니다 — `named named`, `play and play`처럼요. 이 장은 **모델도 학습도 그대로 두고**, 디코딩 방식만 **carry-over semi-AR + 반복 억제**로 바꿔 그 반복을 없앱니다.

학습이 끝난 모델에 새 샘플러를 적용하면 이런 문장이 나옵니다.

> *"Once upon a time, there was a little girl named Lily. She loved to play outside with her friends. One day, she went to the park and saw a big tree in the sky…"*

반복 없이 인물(Lily)·대화·배경이 이어지는 흐름입니다. 이 장의 한 가지 변화인 샘플러는 두 부분으로 이뤄집니다.

- **carry-over semi-AR** — 이전 step에서 확정한 토큰을 다음 step으로 이어받아, 매 step 전체를 다시 흔들지 않고 점진적으로 문장을 굳혀갑니다.
- **반복 억제** — temperature 0.8 / top-p 0.92 / repetition penalty 1.3 / 인접 동일 토큰 금지를 더해, 생성이 한 단어에 갇히지 않게 합니다.

모델 본체·vocab 2048·30000 step·1/t 시간가중 loss는 Ch 32에서 세운 그대로입니다. 이 장은 오직 *어떻게 뽑아내는가*만 바꿉니다.

## 📊 추적표 (Phase 5 — diffusion 라인)

| 챕터 | 모델·학습 (공통) | 샘플러 | 생성 결과 |
|---|---|---|---|
| Ch 32 (패러다임) | BertForMaskedLM hidden 256/4L (3.79M), ByteLevel BPE 2048, 30000 step, 1/t 시간가중 loss | 기본 confidence remasking | coherent하나 반복 ("They run… They run", "happy. happy") |
| **Ch 33 (샘플러)** | **(Ch 32와 동일)** | **carry-over semi-AR + 반복 억제** | **"…named Lily. She loved to play…" — 반복 없음 (4-gram 약 0.17 → 0.00)** |

모델·토크나이저·학습량·loss는 Ch 32와 **완전히 같습니다**. 이 장이 바꾸는 것은 오직 **생성 샘플러** 하나입니다.

## 🔄 변경점 (Diff from Ch 32)

Ch 32에서 작은 diffusion LM이 이미 영어 동화를 생성했습니다(vocab 2048, 30000 step, 1/t loss). 다만 기본 샘플러라 반복이 남았습니다. 이 장의 변화는 **딱 하나, 샘플러**입니다.

| 항목 | Ch 32 | Ch 33 (이번) | 왜 이렇게 바꾸나 |
|---|---|---|---|
| 모델·vocab·학습·loss | hidden 256/4L, BPE 2048, 30000 step, 1/t | **동일 (그대로 상속)** | 모델 품질은 Ch 32에서 이미 확보(고정-t top-1 acc 약 0.71) |
| **샘플러** | 기본 confidence remasking (일괄 복원) | **carry-over semi-AR + 반복 억제** | 확정 토큰을 다음 step으로 이어받고(carry-over), temperature 0.8 / top-p 0.92 / repetition penalty 1.3 / 인접 동일 토큰 금지로 반복을 제거합니다. |

효과는 숫자로 분명합니다. **같은 모델인데 4-gram 반복률이 약 0.17 → 0.00**으로 떨어집니다. train loss(약 3.6)·고정-t acc(약 0.71)는 Ch 32와 같은 모델·학습 레시피라 비슷하게 나옵니다(run 마다 소폭 편차). *생성 품질은 모델만이 아니라 샘플러가 좌우한다* — 이 장의 한 줄 메시지입니다.

## 📐 Loss 노트 — 흡수형 mask diffusion의 시간가중

이 장의 loss는 Ch 32와 **같습니다** — 흡수형(absorbing) mask diffusion의 $1/t$ 시간가중. 이 장은 모델을 새로 학습하니, 복습 차원에서 *왜 $1/t$인지* 다시 짚습니다. 흡수형 mask diffusion의 변분 하한(NELBO)을 제대로 풀면, 각 시점 $t$의 손실에 **시간에 따라 달라지는 가중치**가 붙습니다.

선형 schedule $\alpha_t = 1 - t$를 쓰면 시각 $t$에서 토큰이 아직 마스크되지 않고 살아 있을 확률이 $\alpha_t$입니다. 연속시간 흡수형 확산의 NELBO를 정리하면 시간가중이

$$ \left|\frac{\alpha'_t}{1-\alpha_t}\right| = \frac{1}{1-(1-t)} = \frac{1}{t} \quad (\alpha'_t=-1) $$

로 깔끔하게 환원됩니다($\alpha'_t = -1$). 즉 **가중치는 $1/t$ 하나로 결정**됩니다.

직관은 이렇습니다. $t$가 작으면 거의 안 가려진 상태라 모델이 풍부한 문맥을 보고 빈칸을 맞히는, 정보가 많은 쉬운 상황입니다. $t$가 크면 대부분 가려져 문맥이 거의 없는 어려운 상황입니다. $1/t$ 가중은 **문맥이 많은 저-$t$ 구간에 더 큰 비중**을 실어, 모델이 "주변을 충분히 본 상태에서 정확히 맞히는" 능력을 우선 학습하게 합니다.

수치로 보면 가중치 차이가 한눈에 들어옵니다. 정답 토큰에 모델이 부여한 확률을 $p$라 할 때, 그 자리의 가중 손실은 $\frac{1}{t}\cdot(-\log p)$입니다.

| $t$ | $1/t$ 가중 | 정답 확률 $p$ | $-\log p$ | 가중 손실 $\frac{1}{t}(-\log p)$ |
|----|----------|------------|----------|------------------------------|
| 0.1 | 10.0 | 0.90 | 0.105 | 1.05 |
| 0.5 | 2.0 | 0.60 | 0.511 | 1.02 |
| 0.9 | 1.11 | 0.30 | 1.204 | 1.34 |

같은 $-\log p$라도 저-$t$일수록 가중이 커져 그 자리에서 틀리면 더 크게 벌점을 받습니다. 문맥이 풍부한데도 틀리는 건 변명의 여지가 없는 실수라는 뜻이지요.

구현은 한 배치 안에서

$$ \frac{1}{t}\cdot\frac{\sum_{\text{mask 자리}} (-\log p)}{L} $$

를 계산하고, 이 값을 배치 평균합니다. 여기서 마스크 자리 CE 합을 시퀀스 길이 $L$로 나누는데, 이 $/L$ 때문에 위 값은 엄밀한 ELBO 그 자체는 아니고 **ELBO를 상수배한 surrogate**입니다. 다만 상수배는 gradient의 크기만 바꾸고 **방향은 그대로**여서, 최적화가 향하는 지점은 동일합니다. 정직하게 말하면 우리가 최소화하는 건 ELBO에 비례하는 양이고, 그 비례 덕분에 학습 방향은 올바릅니다.

## 🔤 토크나이저 노트 — 왜 작은 모델엔 작은 vocab인가

이 장도 Ch 32와 **같은 BPE 2048**을 씁니다. 작은 모델에 작은 vocab이 왜 중요한지 복습합니다. 만약 `bert-base-uncased`의 WordPiece vocab 30522개를 hidden 256짜리 작은 모델에 그대로 붙이면 임베딩 테이블만

$$ 30522 \times 256 \approx 7.8\text{M} $$

개의 파라미터를 차지합니다. 전체가 약 11M인 모델에서 **임베딩이 약 70%**를 먹어버리니, 정작 문맥을 학습하는 Transformer 본체에 쓸 용량이 거의 남지 않습니다. 게다가 출력단의 softmax도 매 자리 30522-way 분류라 학습 신호가 넓게 퍼집니다.

해법은 데이터에 맞는 작은 vocab을 직접 학습하는 것입니다. TinyStories 코퍼스에 ByteLevel BPE로 vocab 2048을 학습하면 임베딩은

$$ 2048 \times 256 \approx 0.5\text{M} $$

로 줄어듭니다. 임베딩 비중이 **70%에서 13.9%**로 내려가고, 모델 전체는 3.79M(Ch 24의 GPT와 같은 급)이 됩니다. 본체 용량이 살아나는 것이지요. 출력 softmax도 2048-way로 좁아져, 같은 step 수로도 훨씬 진한 학습 신호를 받습니다.

개념적으로 토큰화를 비교해 보면, WordPiece 30522는 영어 전반을 넓게 덮느라 흔한 동화 단어도 여러 조각으로 쪼개거나 반대로 희귀 단어를 통째로 들고 있습니다. 반면 TinyStories에 직접 학습한 BPE 2048은 이 코퍼스에 자주 나오는 조각(예: 이름·자주 쓰는 일상 단어의 부분)에 집중되어, 작은 vocab으로도 동화 텍스트를 효율적으로 표현합니다. "전 세계 영어를 위한 큰 사전"이 아니라 "이 데이터를 위한 작은 사전"인 셈입니다.

한 가지 빠뜨리면 안 되는 디테일이 있습니다. mask diffusion은 빈칸을 채우는 방식이라 `[MASK]` 토큰이 반드시 필요한데, BPE를 새로 학습하면 이 special token이 기본으로 들어 있지 않습니다. 그래서 vocab을 학습할 때 `[PAD]`, `[UNK]`와 함께 `[MASK]`를 special token으로 추가해 줍니다. 이렇게 추가한 `[MASK]`의 id가 forward 노이징에서 토큰을 가릴 때와 샘플러에서 빈칸을 표시할 때 모두 쓰입니다.

## 🛠️ 환경 셋업

T4 GPU에서 `fp16`로 학습합니다(bf16·flash-attention 불가).

In [1]:
%pip install -q -U transformers tokenizers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/50.1 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.1/50.1 MB 229.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.1/50.1 MB 229.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.1/50.1 MB 229.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.2 MB/s eta 0:00:00


In [2]:
import math, time, torch
import torch.nn.functional as F
from datasets import load_dataset

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
print("torch", torch.__version__, "| device", device, "| fp16", USE_FP16)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch 2.11.0+cu128 | device cuda | fp16 True
GPU: Tesla T4


## 🚀 실습

### 1. TinyStories 로드 (Ch 24/26과 같은 데이터)

In [3]:
raw_train = load_dataset("roneneldan/TinyStories", split="train[:100000]")
raw_val   = load_dataset("roneneldan/TinyStories", split="validation[:500]")
print(raw_train)
print(raw_val[0]["text"][:160])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

...

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 100000
})
Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After


### 2. TinyStories에 BPE 2048 직접 학습 + `[MASK]`

작은 모델에 맞춰 vocab을 직접 학습합니다. `[MASK]`를 special token으로 더해 흡수형 마스킹에 씁니다.

In [4]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast

VOCAB = 2048
def corpus_iter(bs=1000):
    for i in range(0, len(raw_train), bs):
        yield raw_train[i:i+bs]["text"]

_tk = Tokenizer(models.BPE(unk_token="[UNK]"))
_tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
_tk.decoder = decoders.ByteLevel()
_trainer = trainers.BpeTrainer(vocab_size=VOCAB, special_tokens=["[PAD]", "[UNK]", "[MASK]"])
_tk.train_from_iterator(corpus_iter(), trainer=_trainer)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=_tk, pad_token="[PAD]", unk_token="[UNK]", mask_token="[MASK]")
print("vocab_size :", tokenizer.vocab_size)
print("mask_id    :", tokenizer.mask_token_id, "| pad_id:", tokenizer.pad_token_id)
print("sample tok :", tokenizer.tokenize("Once upon a time there was a little cat.")[:14])

vocab_size : 2048
mask_id    : 2 | pad_id: 0
sample tok : ['ĠOnce', 'Ġupon', 'Ġa', 'Ġtime', 'Ġthere', 'Ġwas', 'Ġa', 'Ġlittle', 'Ġcat', '.']


### 3. 토큰화 + `group_texts` (BLOCK_SIZE=128)

In [5]:
BLOCK_SIZE = 128
def tok_fn(b):
    return tokenizer(b["text"], add_special_tokens=False)
tt = raw_train.map(tok_fn, batched=True, remove_columns=raw_train.column_names, desc="tok train")
tv = raw_val.map(tok_fn, batched=True, remove_columns=raw_val.column_names, desc="tok val")

def group_texts(b):
    cat = sum(b["input_ids"], [])
    n = (len(cat) // BLOCK_SIZE) * BLOCK_SIZE
    return {"input_ids": [cat[i:i+BLOCK_SIZE] for i in range(0, n, BLOCK_SIZE)]}
lm_train = tt.map(group_texts, batched=True, remove_columns=tt.column_names, desc="group train")
lm_val   = tv.map(group_texts, batched=True, remove_columns=tv.column_names, desc="group val")
print(f"train chunks {len(lm_train):,} | val {len(lm_val):,} | approx {len(lm_train)*BLOCK_SIZE/1e6:.2f}M tokens")

tok train:   0%|          | 0/100000 [00:00<?, ? examples/s]

tok val:   0%|          | 0/500 [00:00<?, ? examples/s]

group train:   0%|          | 0/100000 [00:00<?, ? examples/s]

group val:   0%|          | 0/500 [00:00<?, ? examples/s]

train chunks 189,030 | val 853 | approx 24.20M tokens


### 4. Diffusion collator — 매 배치 가변 비율 마스킹

`t ~ U(0.02, 1)`로 마스킹 비율을 뽑고(하한 절단), 가린 자리만 학습 신호로 둡니다.

In [6]:
class DiffusionCollator:
    def __init__(self, tok, eps=0.02, seed=SEED):
        self.mask_id = tok.mask_token_id
        self.eps = eps
        self.gen = torch.Generator().manual_seed(seed)   # Trainer seed 와 분리
    def __call__(self, examples):
        ids = torch.tensor([e["input_ids"] for e in examples], dtype=torch.long)
        B, L = ids.shape
        t = torch.rand(B, generator=self.gen) * (1.0 - self.eps) + self.eps
        mask = torch.rand(B, L, generator=self.gen) < t.unsqueeze(1)
        no = ~mask.any(dim=1)
        if no.any():
            j = torch.randint(0, L, (int(no.sum()),), generator=self.gen)
            mask[no, j] = True
        inp = ids.clone(); inp[mask] = self.mask_id
        lab = ids.clone(); lab[~mask] = -100
        return {"input_ids": inp, "attention_mask": torch.ones(B, L, dtype=torch.long),
                "labels": lab, "t": t}
coll = DiffusionCollator(tokenizer)
print("collator ready, mask_id =", coll.mask_id)

collator ready, mask_id = 2


### 5. 작은 BERT-MLM 모델 (Ch 24와 동급, 본체는 그대로)

In [7]:
from transformers import BertConfig, BertForMaskedLM
cfg = BertConfig(vocab_size=tokenizer.vocab_size, hidden_size=256, num_hidden_layers=4,
                 num_attention_heads=4, intermediate_size=1024,
                 max_position_embeddings=BLOCK_SIZE, pad_token_id=tokenizer.pad_token_id)
model = BertForMaskedLM(cfg).to(device)
np_ = model.num_parameters()
emb = tokenizer.vocab_size * cfg.hidden_size
print(f"#params {np_/1e6:.2f}M | embedding share {emb/np_:.1%}  (Ch32: ~70%)")

#params 3.79M | embedding share 13.9%  (Ch32: ~70%)


### 6. 시간가중 `1/t` 손실로 30000 step 학습

In [8]:
from transformers import Trainer, TrainingArguments

class DiffusionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        t = inputs["t"]; labels = inputs["labels"]
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        B, L, V = out.logits.shape
        per = F.cross_entropy(out.logits.view(-1, V), labels.view(-1),
                              ignore_index=-100, reduction="none").view(B, L)
        loss = ((per.sum(dim=1) / L) / t.to(per.dtype)).mean()
        return (loss, out) if return_outputs else loss

args = TrainingArguments(
    output_dir="./out33", max_steps=30000,
    per_device_train_batch_size=64, per_device_eval_batch_size=64,
    learning_rate=3e-4, weight_decay=0.01, warmup_steps=500,
    lr_scheduler_type="cosine", max_grad_norm=1.0, fp16=USE_FP16,
    logging_steps=250, eval_strategy="steps", eval_steps=2000, save_strategy="no",
    report_to="none", label_names=["labels"], remove_unused_columns=False, seed=SEED)

trainer = DiffusionTrainer(model=model, args=args, train_dataset=lm_train,
                           eval_dataset=lm_val, data_collator=coll)
t0 = time.time(); r = trainer.train(); el = (time.time()-t0)/60
print(f"\n=== summary ===\nelapsed {el:.2f} min | step {r.global_step} | train_loss {r.training_loss:.4f}")
print(f"random baseline ln(V) = {math.log(tokenizer.vocab_size):.4f}")
if torch.cuda.is_available():
    print(f"peak VRAM {torch.cuda.max_memory_allocated()/1024**2:.0f} MiB")

Step,Training Loss,Validation Loss
2000,5.886616,5.830340
4000,5.150312,4.844966
6000,3.899159,3.600897
8000,3.594577,3.413167
10000,3.392063,3.102043
12000,3.284058,3.002595
14000,3.200703,2.916767
16000,3.159459,2.870682
18000,3.084729,2.830795
20000,3.045114,2.775763



=== summary ===
elapsed 16.68 min | step 30000 | train_loss 3.6107
random baseline ln(V) = 7.6246
peak VRAM 627 MiB


### 7. carry-over 샘플러로 생성

전부 `[MASK]`에서 시작해 블록 단위로 채웁니다. 기본값은 반복억제 설정(temperature 0.8 · top_p 0.92 · rep penalty 1.3 · 인접중복 금지)입니다.

In [9]:
@torch.no_grad()
def generate(model, length=128, block=32, temperature=0.8, top_p=0.92, top_k=0,
             rep_penalty=1.3, no_immediate_repeat=True, prompt_ids=None):
    """carry-over semi-AR + 반복 억제(rep penalty / 인접중복 금지 / top-p)."""
    model.eval()
    mask_id = tokenizer.mask_token_id
    x = torch.full((1, length), mask_id, dtype=torch.long, device=device)
    fixed = torch.zeros(length, dtype=torch.bool, device=device)
    if prompt_ids is not None:
        p = torch.tensor(prompt_ids[:length], device=device)
        x[0, :len(p)] = p; fixed[:len(p)] = True
    nblocks = (length + block - 1) // block
    for b in range(nblocks):
        lo, hi = b * block, min((b + 1) * block, length)
        steps = hi - lo
        for s in range(steps):
            logits = model(input_ids=x).logits[0].float()        # (L, V)
            logits[:, mask_id] = -1e9
            # 반복 패널티: 이미 확정된 토큰들의 로짓을 깎음
            if rep_penalty and rep_penalty != 1.0:
                comm = x[0][x[0] != mask_id]
                if comm.numel() > 0:
                    u = torch.unique(comm)
                    col = logits[:, u]
                    logits[:, u] = torch.where(col > 0, col / rep_penalty, col * rep_penalty)
            # 인접중복 금지: 각 자리에서 '왼쪽 토큰과 같은 토큰' 예측 차단
            if no_immediate_repeat:
                left = torch.roll(x[0], 1); left[0] = mask_id
                valid = left != mask_id
                logits[valid, left[valid]] = -1e9
            probs = (logits / max(temperature, 1e-6)).softmax(-1)
            if top_k and top_k > 0:
                kth = probs.topk(top_k, dim=-1).values[:, -1, None]
                probs = probs.masked_fill(probs < kth, 0.0)
            if top_p and top_p < 1.0:
                sp, si = probs.sort(dim=-1, descending=True)
                rm = (sp.cumsum(-1) - sp) > top_p
                sp = sp.masked_fill(rm, 0.0)
                probs = torch.zeros_like(probs).scatter(-1, si, sp)
            probs = probs / probs.sum(-1, keepdim=True).clamp_min(1e-9)
            pred = torch.multinomial(probs, 1).squeeze(-1)
            conf = probs.gather(-1, pred.unsqueeze(-1)).squeeze(-1)
            cur = (x[0] == mask_id) & (~fixed)
            cur[:lo] = False; cur[hi:] = False
            nleft = int(cur.sum())
            if nleft == 0: break
            nreveal = nleft if s == steps - 1 else max(1, nleft // (steps - s))
            cc = conf.clone(); cc[~cur] = -1e9
            idx = cc.topk(nreveal).indices
            x[0, idx] = pred[idx]
    return tokenizer.decode(x[0], skip_special_tokens=True)

pid = tokenizer("Once upon a time", add_special_tokens=False)["input_ids"]
torch.manual_seed(SEED)
print("=== unconditional (all-[MASK] -> generate, default sampler) ===")
for i in range(3):
    print(f"[{i}] {generate(model)[:340]}")
print("\n=== conditional (prompt 'Once upon a time' fixed) ===")
for i in range(3):
    print(f"[{i}] {generate(model, prompt_ids=pid)[:340]}")

=== unconditional (all-[MASK] -> generate, default sampler) ===


[0]  says, "We can play with our toys. We have fun."

They get back. They sit on the slide and go to the park. They see a big slide in their rooms. They are happy. They want to play with their dolls.
 
"Can we play outside with it?" Lily asks. She is sad and scared.

Lily does not cry. But she does not listen to her mom or help them. She take


[1]  it."
 
Tom and her Sam go back to the park. He said, "Yes, you can have a great idea!" Tom smiled and replied, "Of course I'm going to play with me?" The little boy thought, "I want to share my game too much fun! You don't like this ball."

Tim nodded and said, "That's very good friend. I love your toy, Tim. We will put them back togethe


[2]  

One day, he saw a big tree in the woods. He wanted to see it was soldier and decided to find something new. The bird felt very happy that he couldn't believe it again! Once upon a time there lived an old page. It was a little boy named Timmy. Billy had a special toy. Once upon a time, there was a little boy named Timmy. Timmy loved to 

=== conditional (prompt 'Once upon a time' fixed) ===


[0]  Once upon a time, there was a little girl named Lily. She loved to play with her toys and wear books. One day, she decided to walk in the park. Her mom said, "Can we go on my closet?" Lily asked her mom if they could march it up. 

Lily's mom told her that she had a new closet. They were very happy and didn't know again. 

One night, Lil


[1]  Once upon a time, there was a little girl named Lily. She loved to play with her toys and have some fun. One day, she went outside to the park with her friends. They played together all day long, but they could not get it on the swings every night. 

Lily's mom said, "I want to go inside now." Her friend replied, "I'm sorry, Lily! I am s


[2]  Once upon a time, there was a little girl. She loved to play with her new friends. One day, she went for a walk in the park and saw something special. It was a big bag of vegetables and had a shiny toy box. Lily thought it looked so pretty! 

The little girl was very excited because she couldn't wait. She wanted to go home, but her mom t


## 🔬 해부 — 모델이 정말 조건부를 배웠나 (눈대중 말고 숫자로)

직전 장에서는 생성 결과를 그냥 눈으로 읽고 "garbage"라고 판정했습니다. 마침표와 `the`, `and`, `was`만 반복되는 문장을 보고 망가졌다고 결론을 내린 셈인데, 이런 눈대중에는 두 가지 함정이 있습니다. 하나는 운 나쁜 샘플 하나에 휘둘릴 수 있다는 것이고, 다른 하나는 "망가졌다"와 "잘된다" 사이의 정도를 잴 수 없다는 것입니다. 8000 step과 30000 step 중 어느 쪽이 얼마나 더 나아졌는지를 눈으로는 구분하기 어렵습니다.

그래서 이 장에서는 세 개의 숫자로 진단합니다. 각 숫자는 서로 다른 질문에 답합니다.

- **고정-t top-1 accuracy**: 모델 자체가 조건부 $p(x_i \mid x_{\setminus \text{mask}})$를 배웠는가
- **KL(생성 ‖ 유니그램)**: 생성 분포가 단순 빈도(유니그램)를 넘어선 구조를 가졌는가
- **4-gram 반복률**: 샘플러가 같은 조각을 되풀이하지 않는가

### 진단 1 — 고정-t top-1 accuracy: "모델 자체"를 떼어내 보기

생성이 망가지는 원인은 크게 둘로 나뉩니다. 모델이 조건부 구조를 못 배웠을 수도 있고, 모델은 멀쩡한데 샘플러(디코딩 절차)가 망쳤을 수도 있습니다. 생성 결과만 보면 이 둘이 뒤섞여서 책임을 가릴 수 없습니다.

고정-t accuracy는 샘플러를 완전히 배제하고 모델만 떼어내 평가합니다. 검증 문장 하나를 가져와 토큰의 **15%만**($t = 0.15$ 고정) `[MASK]`로 가린 뒤, 모델이 그 마스크 자리에서 내놓는 top-1 예측이 원래 토큰과 일치하는 비율을 셉니다. 여러 step에 걸쳐 점진적으로 채우는 생성 절차가 아니라 단 한 번의 forward로 끝나므로, 디코딩 노이즈가 끼어들 여지가 없습니다. 순수하게 "주변 문맥이 주어졌을 때 가린 자리를 맞히는 능력", 즉 조건부 분포 학습 정도만 측정합니다.

학습이 진행되며 이 숫자가 어떻게 움직였는지 봅니다.

| 학습 step | 고정-t(0.15) top-1 accuracy |
|---|---|
| 8000 step (학습 도중 관측) | 약 0.26 |
| 30000 step | **약 0.71** |

만약 모델이 유니그램 marginal만 외운 채 collapse했다면 이 정확도는 0에 가깝습니다. 가린 자리에 문맥과 무관하게 늘 `the`나 마침표만 찍을 테니까요. 약 0.26은 이미 collapse 상태를 벗어났다는 신호이고, 약 0.71은 가려진 자리 10개 중 약 7개를 양방향 문맥만 보고 정확히 복원했다는 뜻입니다. 모델이 "이 앞뒤 문맥이면 여기 올 단어는 이것"이라는 조건부 구조를 실제로 학습했다는 직접 증거입니다.

여기서 한 가지 짚을 점이 있습니다. top-1 accuracy 약 0.71은 곧바로 생성 품질을 뜻하지 않습니다. 생성은 마스크 비율 $t$가 1(전부 가림)에서 시작해 0까지 내려가는 훨씬 어려운 조건이고, 한 번이 아니라 수십 번의 forward를 누적합니다. 그래도 "쉬운 조건(15%만 가림)에서조차 못 맞히면 어려운 생성은 가망이 없다"는 점에서, 이 숫자는 모델 용량에 대한 하한 진단으로 충분합니다.

### 진단 2 — KL(생성 ‖ 유니그램): 빈도를 넘어선 구조인가

collapse한 모델의 가장 교묘한 점은 "그럴듯하게 흔한 단어"를 내놓는다는 것입니다. `the`, `and`, 마침표는 코퍼스에서 원래 자주 등장하므로, 이것들만 반복해도 개별 토큰 빈도는 코퍼스와 비슷해 보입니다. 이 함정을 잡으려면 생성된 토큰의 분포가 코퍼스의 유니그램 분포와 얼마나 **다른지**를 재야 합니다.

$$\mathrm{KL}(P_{\text{gen}} \,\|\, P_{\text{unigram}}) = \sum_{v} P_{\text{gen}}(v) \log \frac{P_{\text{gen}}(v)}{P_{\text{unigram}}(v)}$$

직관은 거꾸로 읽어야 합니다. 모델이 유니그램 marginal만 학습했다면 생성 분포가 코퍼스 빈도를 그대로 따라가므로 이 KL은 0 근처로 떨어집니다. 반대로 모델이 문맥에 따라 단어를 가려 쓰는 조건부 구조를 배웠다면, 특정 문맥에서 특정 단어를 몰아 쓰게 되어 전체 생성 분포가 평평한 유니그램에서 멀어집니다. 그래서 **KL이 클수록 구조를 배운 것**입니다.

| 측정 | KL(생성 ‖ 유니그램) |
|---|---|
| 교정된 모델 (30000 step) | **약 0.58** |

약 0.58은 생성 토큰 분포가 단순 빈도표와 뚜렷이 갈라졌다는 뜻입니다. collapse 모델이라면 여기서 0에 가까운 값이 나왔을 것입니다. 진단 1의 top-1 accuracy가 "마스크 자리를 맞힌다"는 국소적 증거라면, 이 KL은 "생성 전체가 유니그램 흉내를 넘어섰다"는 분포 차원의 증거입니다. 두 숫자가 같은 결론을 다른 각도에서 받쳐 줍니다.

### 진단 3 — 4-gram 반복률: 샘플러가 같은 조각을 맴도는가

앞의 두 숫자가 모델을 평가했다면, 이번 숫자는 샘플러를 평가합니다. 모델이 조건부를 잘 배웠더라도 디코딩 절차가 서툴면 생성이 같은 4-단어 조각을 무한히 되풀이하는 루프에 빠집니다. 4-gram 반복률은 생성문에서 한 번 이상 중복 등장한 4-gram의 비율로, 이런 맴돎을 직접 잡아냅니다.

| 샘플러 | 4-gram 반복률 |
|---|---|
| 반복 억제 없음 | 약 0.17 |
| 반복 억제 적용 | **0.00** |

반복 억제가 없을 때의 약 0.17은 생성문 4-gram의 약 17%가 어딘가에서 다시 등장한다는 뜻으로, 자연스러운 문장이라기엔 높습니다. 반복 억제를 켜면 0.00으로 떨어집니다. 여기서 쓴 장치는 세 가지입니다. 이미 사용한 토큰의 로짓을 깎는 repetition penalty(1.3), 바로 왼쪽 토큰과 동일한 예측을 막는 no-immediate-repeat, 그리고 분포 꼬리를 잘라 안정화하는 temperature 0.8 + top-p 0.92입니다.

한 가지 균형을 분명히 해 둡니다. 4-gram 반복률 0.00은 그 자체로 "좋은 글"의 증거가 아닙니다. 무작위로 단어를 뽑아도 반복은 0이 나올 수 있으니까요. 이 숫자는 진단 1(조건부 학습됨)과 진단 2(구조 있음)가 이미 통과한 **뒤에** 읽어야 의미가 있습니다. 모델은 구조를 배웠고, 샘플러는 그 구조를 같은 조각으로 망치지 않는다 — 세 숫자를 함께 읽을 때 비로소 이 결론이 섭니다.

### 세 숫자를 한 장에 — collapse ablation 대비

레시피를 일부러 되돌려 망가뜨린 collapse 모델(아래 삽질 코너에서 재현)과 이 장의 모델을 같은 잣대로 나란히 놓으면, 레시피가 무엇을 바꾸는지 한눈에 들어옵니다.

| 진단 지표 | collapse 상태 (예상) | 교정된 모델 (실측) | 무엇을 증명하나 |
|---|---|---|---|
| 고정-t(0.15) top-1 accuracy | ≈0 | 약 0.26 → **약 0.71** | 모델이 조건부를 학습 |
| KL(생성 ‖ 유니그램) | ≈0 | **약 0.58** | 생성이 유니그램 흉내를 넘어섬 |
| 4-gram 반복률 | 높음 | 약 0.17 → **0.00** | 샘플러가 같은 조각을 안 맴돎 |

세 숫자는 같은 이야기를 세 각도에서 합니다. top-1 accuracy 약 0.71은 모델이 가려진 자리를 양방향 문맥으로 복원한다는 것을, KL 약 0.58은 그 결과가 단순 빈도표가 아니라 문맥 의존 구조라는 것을, 4-gram 0.00은 그 구조가 디코딩 단계에서 반복 루프로 무너지지 않는다는 것을 보여줍니다. collapse ablation이 "무엇이 망가지나"를 보여준다면, 이 숫자들은 "이 모델이 정말 조건부 구조를 배웠나"를 답합니다.

이렇게 진단된 모델이 실제로 어떤 문장을 쓰는지는 다음 셀의 생성 예시에서 직접 확인합니다.

In [10]:
# 정량 진단 (모델 자체 — 샘플러 무관)
g = torch.Generator().manual_seed(0)
def fixed_t_acc(t_val=0.15, n=128):
    cor = tot = 0
    for ex in lm_val.select(range(min(n, len(lm_val)))):
        ids = torch.tensor(ex["input_ids"])
        m = torch.rand(len(ids), generator=g) < t_val
        if not m.any(): m[0] = True
        inp = ids.clone(); inp[m] = tokenizer.mask_token_id
        with torch.no_grad():
            pr = model(inp.unsqueeze(0).to(device)).logits[0].argmax(-1).cpu()
        cor += (pr[m] == ids[m]).sum().item(); tot += int(m.sum())
    return cor / tot
print(f"[진단] 고정-t(0.15) top-1 accuracy = {fixed_t_acc():.3f}")

# 반복도 측정: 생성문의 4-gram 중복 비율 (낮을수록 좋음)
def rep4(text):
    toks = text.split()
    if len(toks) < 5: return 0.0
    grams = [tuple(toks[i:i+4]) for i in range(len(toks)-3)]
    return 1.0 - len(set(grams)) / len(grams)
import statistics
bestB = [generate(model, prompt_ids=pid, temperature=0.8, top_p=0.92, rep_penalty=1.3, no_immediate_repeat=True) for _ in range(5)]
baseA = [generate(model, prompt_ids=pid, temperature=0.7, top_k=40, top_p=1.0, rep_penalty=1.0, no_immediate_repeat=False) for _ in range(5)]
print(f"[진단] 4-gram 반복률  A(기존)={statistics.mean(map(rep4,baseA)):.3f}  B(반복억제)={statistics.mean(map(rep4,bestB)):.3f}  (낮을수록 좋음)")

# 진단 2 — KL(생성 ‖ 유니그램): 생성 분포가 코퍼스 빈도(유니그램)를 넘어선 구조인가
import numpy as np
V = tokenizer.vocab_size
uni = np.zeros(V)
for ex in lm_val.select(range(min(500, len(lm_val)))):
    for t in ex["input_ids"]:
        if 0 <= t < V: uni[t] += 1
P_uni = uni + 1e-6; P_uni /= P_uni.sum()
torch.manual_seed(123)
kl_samples = [generate(model, prompt_ids=pid, temperature=0.8, top_p=0.92,
                       rep_penalty=1.3, no_immediate_repeat=True) for _ in range(20)]
gen_cnt = np.zeros(V)
for text in kl_samples:
    for t in tokenizer.encode(text):
        if 0 <= t < V: gen_cnt[t] += 1
P_gen = gen_cnt + 1e-6; P_gen /= P_gen.sum()
kl_gen_uni = float((P_gen * np.log(P_gen / P_uni)).sum())
print(f"[진단] KL(생성 ‖ 유니그램) = {kl_gen_uni:.3f}  (클수록 유니그램 흉내를 넘어선 구조)")

[진단] 고정-t(0.15) top-1 accuracy = 0.712


[진단] 4-gram 반복률  A(기존)=0.174  B(반복억제)=0.000  (낮을수록 좋음)


[진단] KL(생성 ‖ 유니그램) = 0.577  (클수록 유니그램 흉내를 넘어선 구조)


## 🛠️ 변형 — 샘플러만 바꿔보기

여기서는 **모델을 다시 학습하지 않습니다.** 방금 30000 step으로 학습한 그 가중치를 그대로 두고, 디코딩(샘플러)만 두 가지로 바꿔 결과가 어떻게 달라지는지 봅니다. 학습이 끝난 모델은 "각 자리에 어떤 토큰이 올 확률이 높은가"를 알고 있을 뿐이고, 그 확률 분포에서 실제 문장을 어떻게 뽑아낼지는 전적으로 샘플러의 몫입니다.

### 비교 A — 반복 억제 없는 단순 샘플러

가장 소박하게, 매 step 모델이 내놓은 로짓을 그대로 받아 확률이 높은 토큰부터 채워 넣습니다. 온도도 1.0, top-p도 없고, 같은 토큰을 다시 써도 막지 않습니다. 이렇게 하면 모델이 한번 "안전한" 토큰(자주 등장하는 단어나 구두점)에 높은 확률을 주기 시작했을 때 그 토큰이 계속 반복되기 쉽습니다.

실제로 이 샘플러의 **4-gram 반복률은 약 0.17** 입니다. 생성문을 읽어 보면 같은 짧은 구절이 돌림노래처럼 되풀이됩니다(실제 출력은 아래 비교 셀에서 확인합니다). 모델이 틀려서가 아닙니다. 다음에 올 가장 그럴듯한 토큰을 매번 충실히 고르다 보니, 한번 안전한 구절을 고르면 그 구절이 국소적으로 가장 그럴듯한 선택이 되어 버리는 함정에 빠진 것입니다.

### 비교 B — 반복 억제 샘플러 (carry-over semi-AR)

같은 모델, 같은 가중치에 다음 장치들을 더합니다.

- **temperature 0.8** — 로짓을 살짝 날카롭게 해 너무 평평한 분포에서 엉뚱한 토큰이 튀는 걸 줄입니다.
- **top-p 0.92** — 누적 확률 0.92 안쪽의 토큰만 후보로 남기는 nucleus sampling으로 꼬리쪽 저확률 토큰을 잘라냅니다.
- **repetition penalty 1.3** — 이미 써 버린 토큰의 로짓을 깎아 같은 단어가 다시 뽑힐 확률을 낮춥니다.
- **no immediate repeat** — 바로 왼쪽 토큰과 똑같은 예측을 차단해 "the the", "sorry sorry" 같은 인접 중복을 원천 봉쇄합니다.

결과는 극적입니다. **4-gram 반복률이 약 0.17에서 0.00으로 떨어집니다.** 앞서 돌림노래처럼 반복되던 문장이 사라지고, 인물과 대화가 있는 이야기가 나옵니다(아래 비교 셀의 실제 출력 참고).

### carry-over의 의미 — 확정한 토큰은 건드리지 않는다

이 샘플러의 핵심은 이름에 들어 있는 **carry-over** 입니다. block(32 토큰) 단위로 왼쪽에서 오른쪽으로 진행하면서, 매 step 전체를 다시 예측하되 **이미 확정한(reveal한) 토큰은 다음 step으로 그대로 이월하고 절대 바꾸지 않습니다.** 새로 채울 자리는 mask로 남은 위치 중 모델이 가장 확신하는 곳부터 고신뢰 순으로 확정합니다.

이것이 Ch 32 기본 샘플러가 남긴 반복의 직접적 교정점입니다. Ch 32의 "저신뢰 재마스킹"(기본) 샘플러는 방금 채운 토큰조차 신뢰도가 낮으면 도로 `[MASK]`로 지웠습니다. 디코딩이 단조적이지 않으니 어렵게 만든 문맥이 매 step 허물어졌습니다. carry-over는 한번 확정한 토큰을 불변으로 두어, 오른쪽으로 갈수록 확정된 왼쪽 문맥이 차곡차곡 쌓이게 만듭니다. block 마지막 step에서는 남은 자리를 전부 확정해 빈칸 없이 블록을 마무리합니다.

### "모델이 좋아야 샘플러가 산다"

여기서 한 가지를 분명히 해 둘 필요가 있습니다. 반복 억제 샘플러가 **collapse한 모델**(삽질 코너에서 vocab·step을 되돌려 망가뜨린 ablation)을 살려내는 건 **아닙니다.** 그렇게 유니그램만 외운 모델에 이 샘플러를 붙였다면 반복은 줄었겠지만 여전히 의미 없는 문장만 나왔을 것입니다. 유니그램 marginal만 학습한 모델에는 애초에 뽑아낼 조건부 구조가 없기 때문입니다.

이번 장에서 반복 억제가 효과를 본 건 **모델이 먼저 제대로 학습됐기 때문**입니다. 고정-t(0.15) top-1 accuracy가 약 0.26에서 약 0.71로 오른, 조건부 구조를 실제로 익힌 모델 위에서만 샘플러의 미세 조정이 빛을 봅니다. 샘플러는 좋은 모델의 잠재력을 끌어낼 뿐, 없는 능력을 만들어 내지는 못합니다.

### block 크기와 온도의 trade-off

마지막으로 두 하이퍼파라미터의 균형을 짚어 둡니다.

- **block 크기** — 작게 잡으면(예: 8) 왼→오 진행이 잘게 쪼개져 더 autoregressive에 가까워지고 국소적으로 일관되지만, 한 번에 보는 미래 문맥이 좁아 전역 구성이 약해집니다. 크게 잡으면 양방향 문맥을 넓게 쓰지만 한 블록 안에서 동시에 채워야 할 자리가 많아 거칠어질 수 있습니다. 본 장은 32로 두었습니다.
- **temperature** — 낮추면(0.8) 안전하고 매끄럽지만 단조로워지고, 높이면 다양해지지만 엉뚱한 토큰이 늘어 문장이 깨질 위험이 커집니다.

정답은 한 점이 아니라 "얼마나 안전하게 vs 얼마나 다채롭게"의 저울질입니다. 작은 모델일수록 안전한 쪽으로 살짝 기울여 두는 편이 읽을 만한 결과를 줍니다.

In [11]:
print("=== 샘플러 sweep (같은 학습 모델, 조건부 'Once upon a time') ===")
pid = tokenizer("Once upon a time", add_special_tokens=False)["input_ids"]
configs = [
    ("A) 기존 temp0.7/topk40 (반복억제 없음)", dict(temperature=0.7, top_k=40, top_p=1.0, rep_penalty=1.0, no_immediate_repeat=False)),
    ("B) rep1.3 + 인접금지 + topp0.92",        dict(temperature=0.8, top_p=0.92, rep_penalty=1.3, no_immediate_repeat=True)),
    ("C) rep1.2 + temp0.9 + topp0.95",         dict(temperature=0.9, top_p=0.95, rep_penalty=1.2, no_immediate_repeat=True)),
    ("D) B + block16 (더 촘촘)",                dict(temperature=0.8, top_p=0.92, rep_penalty=1.3, no_immediate_repeat=True, block=16)),
]
torch.manual_seed(SEED)
for name, kw in configs:
    print(f"\n----- {name} -----")
    for i in range(2):
        print(f"[{i}] {generate(model, prompt_ids=pid, **kw)[:360]}")

=== 샘플러 sweep (같은 학습 모델, 조건부 'Once upon a time') ===

----- A) 기존 temp0.7/topk40 (반복억제 없음) -----


[0]  Once upon a time, there was a little girl named Lily. She loved to play with her toys with her friends. One day, she went to the park with her mommy and daddy. Lily was so excited because she wanted to play outside in the park and asked her mommy.

Lily's mommy told her that she was going to play with her toys. Lily was so excited that she was going to play


[1]  Once upon a time, there was a little girl named Lily. She loved to play with her toys with her toys. One day, Lily went to the park with her mommy and daddy. She saw a big, red ball with a ball. Lily was so happy that she couldn't reach it.

Lily's mommy gave her a big red ball and started to play with it. Lily was so happy that she couldn't reach the ball 

----- B) rep1.3 + 인접금지 + topp0.92 -----


[0]  Once upon a time, there was a little girl named Lily. She loved to play with her toys and make pretty flowers. One day, she went on the swings in the park when she saw a big red tree. It looked so tall that it wanted to wast it! 

Lily's mom said, "I want to see what I found my ball." Her mommy replied, "Yes, let's go home!"
 

Her mom smiled and said, "I'm


[1]  Once upon a time, there was a little girl. She loved to play with her toys and watch other animals. One day, she would go to the park when it saw a big red tree. It looked pretty and wanted to catch it so much that Lily didn't know! 

Lily's mom said, "I'm going to throw your ball, please?" Her mom replied, "Yes, you can help me." 

Her mom replied, "Why do

----- C) rep1.2 + temp0.9 + topp0.95 -----


[0]  Once upon a time, there was a little girl named Lily. She loved to play with her toys and leaves. One day, she saw a big forest with her mommy. She was very excited because she wanted to see it. She asked her, "Can I play in the distance you?" 

Lily's mom said, "I'm sorry, but we don't have to play with your toy masks." 

Her mom replied, "No, we can be ca


[1]  Once upon a time, there was an old man called Jake. He loved to play in the forest and explore his worlds. One day, he went on a walk with his mom and dad. The boy was so excited that he could see what it was for. 

He saw something shiny! He had never seen it. It looked like a new toy. He thought this was special and looked inside. He wanted a big car but 

----- D) B + block16 (더 촘촘) -----


[0]  Once upon a time, there was an old girl named Lily. She loved to play with her toys. One day, she was playing in the park and wanted to go on it from her house. 

Lily's mom said, "I want you go for help me first too." Her dad replied, "Sure, Mommy! I can have some cookies and having fun." 

Her friends were so happy that they went to the store. They played


[1]  Once upon a time, there was an old girl named Lily. She loved to play with her toys and make things in the park. One day, she went on a new house with her mom's toy. 

Lily wanted to buy some candy so he could fix it. Her mommy said, "I want you to use this money for dinner instead." 

Her mommy replied, "Yes, we can help me clean your room together!" So th


## 🆚 Autoregressive(Ch 24) vs Diffusion(이 장)

같은 데이터(TinyStories), 같은 토크나이저(ByteLevel BPE vocab 2048), 거의 같은 규모(약 3.7M-3.8M params)로 만든 두 언어 모델을 나란히 놓고 보면, 생성 방식의 차이가 학습과 추론 전반에 어떻게 번지는지가 또렷이 드러납니다.

| 항목 | Autoregressive (Ch 24, GPT) | Diffusion (이 장, mask-diffusion) |
|---|---|---|
| 생성 방향 | 왼→오 단방향, 한 토큰씩 순차 | 전체 mask에서 시작해 양방향 문맥으로 병렬 채움 |
| 학습 감독 | 매 자리마다 다음 토큰 예측 — 모든 자리가 매 step 감독됨 | mask로 가린 자리만 예측 — 가린 자리에서만 학습 신호 |
| 마스킹 | 인과 마스크 고정(왼쪽만 봄) | 매 배치 확률 $t$로 토큰마다 가변 마스킹 |
| 필요 학습량 | 1500 step | 30000 step (약 20배) |
| 생성 비용 | KV-cache로 1패스 | NFE가 대략 생성 길이에 비례, 반복 디코딩 |
| 같은 규모 품질 | 상대적으로 매끄러움 | 같은 규모에서 더 거친 게 정상 |

### 왜 diffusion이 약 20배 더 많은 step을 요구하나

핵심은 **학습 신호의 밀도** 입니다. Autoregressive 모델은 길이 $L$ 문장 한 개에서 $L$개 자리 전부에 대해 "왼쪽 전체 문맥으로 다음 토큰 맞히기" 과제를 풉니다. 한 문장이 곧 $L$개의 감독 신호인 셈이라, 토큰마다 빠짐없이 기울기가 흐릅니다.

Diffusion 모델은 한 배치에서 확률 $t$로 일부 자리만 가립니다. 가리지 않은 자리는 정답이 그대로 입력에 들어 있으니 손실에 기여하지 않고(`labels`를 -100으로 두어 제외), 가린 자리에서만 학습이 일어납니다. 게다가 그 가린 자리도 "어떤 자리가 가려졌는가"가 배치마다 달라지는 양방향 빈칸 채우기여서, 한 토큰을 여러 마스킹 패턴에서 반복적으로 마주쳐야 비로소 안정적으로 익혀집니다. 자리당 감독이 희박하고 과제가 매번 달라지니, 같은 데이터를 더 여러 번 통과시켜야(여기서는 약 20배) 비슷한 수준에 도달합니다.

이건 diffusion의 결함이라기보다 **무엇과 맞바꾸는가** 의 문제입니다. 단방향 순차 예측의 빽빽한 감독을 포기하는 대신, diffusion은 양방향 문맥과 병렬 채움이라는 다른 성질을 얻습니다.

### 생성 비용 — 1패스 vs 길이에 비례한 반복

추론에서도 둘은 갈립니다. Autoregressive는 KV-cache 덕에 이미 생성한 토큰의 계산을 재활용하며 사실상 1패스로 끝까지 흘러갑니다. Diffusion은 전부 `[MASK]`인 상태에서 출발해 매 step 전체를 다시 예측하고 일부만 확정하는 과정을 되풀이하므로, 함수 평가 횟수(NFE)가 대략 생성 길이에 비례합니다. carry-over semi-AR로 block을 잘라 진행해도 step 수가 길이를 따라 늘어나는 구조는 그대로입니다.

### 무엇이 더 거친가 — 그리고 그게 정상인 이유

결과물의 결을 보면, 같은 약 3.7M 규모에서 Ch 24의 GPT는 1500 step만으로도 "there was a girl named Lily" 같은 매끄러운 문장을 냈습니다. 이 장의 diffusion은 30000 step을 들여 인물(Lily, Timmy)·대화·배경이 있는 이야기까지 도달했지만, "big collar tree", "an noise" 같은 자잘한 흠이 남습니다.

이 거칠기는 모델이 잘못 학습됐다는 신호가 아닙니다. **같은 규모에서 diffusion이 autoregressive보다 거친 건 정상** 입니다. 양방향 병렬 채움이라는 더 어려운 과제를, 더 희박한 감독으로, 작은 본체로 풀고 있기 때문입니다. 중요한 건 이 모델이 유니그램 collapse(". the the.. was" 같은 고빈도 토큰 반복)에 빠지지 않고 조건부 구조를 실제로 학습했다는 점입니다. 고정-t(0.15) top-1 accuracy가 약 0.71까지 오르고, 생성 토큰 분포가 코퍼스 유니그램과 뚜렷이 다른(KL 약 0.58) 상태가 그 증거입니다.

### 정리

Autoregressive와 diffusion 중 하나가 일방적으로 우월한 게 아닙니다. AR은 빽빽한 감독과 KV-cache로 작은 규모·짧은 학습에서 유리하고, diffusion은 양방향 문맥과 병렬 생성이라는 다른 길을 택하는 대신 더 많은 학습량과 반복 추론을 치릅니다. 작은 모델·짧은 예산이라는 이 장의 조건에서는 AR이 더 손쉽게 매끄러운 결과를 주지만, diffusion도 레시피만 제대로 잡으면 같은 T4 30분 예산 안에서 충분히 coherent한 이야기를 만들어 낸다는 것을 이 장이 보여 줍니다.

## 📦 등장한 라이브러리 정리

이번 장에서 새로 등장했거나, 익숙한 도구를 새로운 방식으로 쓴 부분만 추렸습니다.

- **`tokenizers` (BPE 직접 학습)** — `bert-base-uncased`의 WordPiece 30522개 대신, TinyStories 코퍼스에 ByteLevel BPE로 vocab 2048개를 직접 학습했습니다. `[PAD]`, `[UNK]`, `[MASK]` 특수 토큰을 더해 흡수형 마스킹에 바로 쓸 수 있게 했습니다. vocab을 줄이자 임베딩 테이블이 차지하던 파라미터 비중이 70%에서 13.9%로 떨어지고, 같은 11M급이던 모델이 본체 위주의 3.79M으로 슬림해졌습니다.

  ```python
  from tokenizers import ByteLevelBPETokenizer

  tok = ByteLevelBPETokenizer()
  tok.train_from_iterator(
      text_iter,                 # TinyStories 텍스트 제너레이터
      vocab_size=2048,
      special_tokens=["[PAD]", "[UNK]", "[MASK]"],
  )
  ```

- **`BertForMaskedLM` (흡수형 diffusion 백본으로 재활용)** — MLM용 모델이지만, 매 배치마다 각 토큰을 확률 $t$로 `[MASK]`로 덮는 forward 노이징과 시간가중 손실을 얹으면 흡수형 mask diffusion 학습기로 그대로 쓸 수 있습니다. 본체는 hidden 256 / 4L / 4H / intermediate 1024 / max_pos 128로 작게 유지했습니다. 키우면 30분 예산을 금세 넘기기 때문입니다.

- **시간가중 NELBO 손실 (`compute_loss` 커스텀)** — 마스크된 자리에만 교차엔트로피를 매기고, 선형 schedule $\alpha_t = 1 - t$에서 시간가중 $1/t$를 곱하는 흡수형 NELBO를 직접 구현했습니다. `labels`는 마스크 자리만 토큰 id, 나머지는 `-100`(`ignore_index`)으로 둡니다.

  ```python
  # mask_logits: 마스크 자리 로짓, mask_labels: 그 자리 정답 id
  ce = F.cross_entropy(mask_logits, mask_labels, reduction="sum")
  loss = (ce / seq_len) / t          # (CE합 / L) / t
  loss = loss.mean()                 # 배치 평균
  ```

- **carry-over semi-AR 샘플러** — 전부 `[MASK]`인 상태에서 시작해, 32 토큰 블록을 왼쪽에서 오른쪽으로 채웁니다. 매 step마다 전체를 예측하되 생성에서 `mask_id` 로짓은 $-\infty$로 막고, 한 번 확정한 토큰은 다시 가리지 않습니다(carry-over). 저신뢰 재마스킹이 없어 디코딩이 단조롭게 진행됩니다. 반복을 누르려고 temperature 0.8, top_p 0.92, repetition penalty 1.3, 그리고 바로 왼쪽 토큰과 같은 예측을 막는 `no_immediate_repeat`을 함께 걸었습니다.

## 🎯 체크포인트 질문

1. 레시피를 되돌리면(vocab 30522 또는 1500 step) train_loss가 6.03 근처에서 멈추고, 이 값이 코퍼스의 유니그램 엔트로피와 거의 같습니다. 이 한 가지 숫자만으로 "모델이 조건부 구조가 아니라 단어 빈도만 외웠다"고 진단할 수 있는 이유는 무엇일까요?

2. vocab을 30522에서 2048로 줄였더니 임베딩이 차지하던 파라미터 비중이 70%에서 13.9%로 떨어졌습니다. 본체 hidden/layer 수는 그대로 두었는데도 생성 품질이 살아난 까닭을 "용량 배분" 관점에서 설명해 보세요.

3. carry-over 샘플러는 한 번 확정한 토큰을 다시 `[MASK]`로 되돌리지 않습니다. Ch 32의 기본 샘플러("저신뢰 재마스킹")는 방금 채운 토큰을 도로 지웠습니다. 이 둘 중 어느 쪽이 디코딩을 비단조로 만들고, 그것이 왜 collapse를 부추기는지 말해 보세요.

4. 같은 3.7M급, 같은 TinyStories인데 AR(Ch 24)은 1500 step, 이번 diffusion은 30000 step이 필요했습니다. 학습 신호가 매 자리에 들어오는지 마스크 자리에만 들어오는지의 차이로 이 20배 격차를 설명해 보세요.

## ❓ FAQ

**Q1. 왜 큰 vocab(30522)이면 collapse하고 작은 vocab(2048)이면 괜찮나요? 단어를 적게 아는 모델이 더 잘 쓴다는 게 직관에 안 맞습니다.**

collapse의 핵심은 어휘량이 아니라 파라미터 배분입니다. vocab 30522짜리 11M 모델은 그중 약 70%가 임베딩 테이블(30522 × hidden)에 묶입니다. 정작 문맥을 읽고 다음 토큰을 추론하는 트랜스포머 본체에 쓸 용량이 거의 남지 않았던 셈입니다. vocab을 2048로 줄이면 임베딩 비중이 13.9%로 내려가고, 같은 예산을 본체가 가져가 조건부 구조를 학습할 여력이 생깁니다. TinyStories는 어휘가 단순한 동화 코퍼스라 2048개로도 표현력이 충분하다는 점도 맞물립니다. 즉 "단어를 적게 안다"가 아니라 "아낀 용량을 문맥 추론에 돌렸다"가 정확한 그림입니다.

**Q2. 그러면 본체(hidden/layer)는 왜 같이 안 키웠나요? 용량이 핵심이라면 본체도 키우면 더 좋지 않나요?**

좋아지긴 하겠지만 T4의 30분 예산을 지킬 수 없습니다. 본체를 키우면 step당 연산이 늘어 같은 step 수를 도는 데 시간이 폭증합니다. 이번 교정의 전략은 "본체는 Ch 24와 동급(3.79M)으로 유지하되, 임베딩에서 빼낸 용량을 본체가 충분히 활용하도록 학습량을 늘리는 것"이었습니다. 실제로 본체는 그대로 두고 max_steps를 (ablation의) 1500이 아닌 30000으로 두어 18.5분에 train_loss 약 3.6까지 내렸습니다. 같은 예산 안에서는 본체 확대보다 학습량 확보가 더 큰 레버였습니다.

**Q3. 왜 diffusion이 AR보다 20배나 많은 step이 필요한가요?**

학습 신호의 밀도가 다릅니다. AR(Ch 24)은 한 문장을 한 번 넣으면 모든 위치에서 "왼쪽 전체 문맥으로 다음 토큰 맞히기" 감독이 동시에 걸립니다. 매 자리가 매번 학습됩니다. 반면 흡수형 diffusion은 매 배치 확률 $t$로 일부 토큰만 `[MASK]`로 덮고, 그 마스크 자리에서만 손실을 받습니다. 마스킹 비율 $t$도 배치마다 달라져서, 같은 문장이라도 어느 자리가 감독될지가 매번 바뀝니다. 결과적으로 자리당 받는 신호가 희박해 같은 수준에 도달하려면 더 많은 step이 필요합니다. Ch 24가 1500 step, 이번 장이 30000 step이었던 약 20배 격차가 이 차이를 보여줍니다.

**Q4. carry-over가 Ch 32의 저신뢰 재마스킹과 구체적으로 뭐가 다른가요?**

방향이 정반대입니다. 저신뢰 재마스킹은 매 step "확신이 약한 자리"를 도로 `[MASK]`로 지우고 다시 예측합니다. 방금 그럴듯하게 채운 토큰이 다음 step에 다시 빈칸이 되는, 비단조(채웠다 지웠다) 디코딩이라 작은 모델에서는 안정점을 못 찾고 고빈도 토큰으로 무너지기 쉽습니다. carry-over는 반대로 한 번 reveal한 토큰을 끝까지 고정합니다. 블록 안에서 고신뢰부터 차례로 확정하고, 확정된 토큰은 이후 step의 입력에서 불변으로 남습니다(carry-over). 디코딩이 단조롭게 한 방향으로만 진행돼 안정적입니다.

```python
# 매 step: 전체 예측 후 생성에서 mask_id는 막고, 고신뢰부터 reveal
logits[..., mask_id] = float("-inf")     # 생성 결과에 [MASK] 못 나오게
revealed[pos] = True                      # 한 번 확정하면
x[pos] = chosen_id                        # 이후 step에서 불변(carry-over)
```

**Q5. 손실의 $1/t$ 시간가중은 왜 필요한가요? 빼면 안 되나요?**

흡수형 NELBO를 제대로 근사하려면 필요합니다. 마스킹 비율이 작을 때($t$가 작을 때)는 가려진 토큰이 몇 개 안 되지만, 그 적은 자리를 맞히는 일이 ELBO에서 차지하는 가중치는 큽니다. 선형 schedule $\alpha_t = 1 - t$에서 이 가중치가 정확히 $1/t$로 나옵니다. 구현은 마스크 자리 CE 합을 길이 $L$로 나눈 뒤 다시 $t$로 나누는 형태입니다.

```python
loss = ((mask_ce_sum / seq_len) / t).mean()   # 1/t 시간가중
```

$1/t$를 빼면 마스킹이 많은(쉬운 신호가 많은) step에 학습이 쏠려 적게 가려진 어려운 경우를 덜 배우게 됩니다. 참고로 `/L`은 ELBO를 상수배한 surrogate라 gradient 방향은 그대로지만, $1/t$는 step별 가중 자체를 바꾸므로 성질이 다릅니다.

**Q6. 학습 때 $t$의 하한(0.02)만 자르고 상한은 1로 그대로 둔 이유가 있나요?**

train과 infer의 조건을 맞추기 위해서입니다. 하한 0.02로 절단하는 건 $t \to 0$에서 $1/t$ 가중이 폭발해 학습이 불안정해지는 것을 막으려는 수치적 안전장치입니다. 반대로 상한을 자르지 않는 이유는, 생성이 전부 `[MASK]`인 상태($t = 1$에 해당)에서 시작하기 때문입니다. 만약 학습에서 $t$ 상한을 0.9 같은 값으로 잘라버리면 모델은 "거의 다 가려진 입력"을 본 적이 없는데 추론에서는 바로 그 상황을 마주합니다. train에서 안 본 분포를 infer에서 요구하면 또 무너지기 쉽습니다. 그래서 상한은 1로 열어 둡니다.

```python
t = torch.rand(batch).clamp_min(0.02)   # 하한만 절단, 상한은 1 그대로
```

**Q7. 생성 결과에 "big collar tree", "an noise" 같은 잔흠이 남는데, 더 키워야 하나요?**

이건 collapse가 아니라 작은 모델의 한계입니다. 3.79M급에 vocab 2048이면 인물(Lily, Timmy)과 대화, 배경이 있는 동화를 만들어 내는 것 자체가 성공 신호입니다. 실제로 고정-$t$(0.15) top-1 정확도가 8000 step의 약 0.26에서 30000 step의 약 0.71까지 올랐고, 반복억제 샘플러로 4-gram 반복률이 약 0.17에서 0.00으로 떨어졌으며, 생성 토큰 분포와 코퍼스 유니그램의 KL이 약 0.58로(collapse면 0에 가깝습니다) 모델이 단순 빈도 모사를 넘어섰음을 보여줍니다. 자잘한 흠은 모델·데이터 규모를 키우면 줄지만, 그건 T4 30분 예산 밖의 이야기입니다. 이 장의 목표는 "작은 diffusion LM이 coherent하게 생성되게 만든다"이고, 그 선은 넘었습니다.

## 🚀 삽질 코너 — "조용한 collapse" 재현하기

이 레시피(작은 vocab·충분한 학습)가 정말 효과의 핵심인지 확인하려면, 일부러 되돌려 보면 됩니다. 두 가지만 건드려도 collapse가 그대로 돌아옵니다.

**되돌리기 ① vocab을 30522로**

```python
# 직접 학습한 BPE 2048 대신 원래 WordPiece 30522로 되돌리면
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # vocab 30522
# → 임베딩이 다시 파라미터의 ~70%를 먹고 본체 용량 고갈
```

같은 11M 안에서 본체가 쓸 용량이 말라, 학습을 돌려도 train_loss가 유니그램 엔트로피 근처(약 6.0)에서 멈춥니다. 생성은 ". the the.. was"처럼 고빈도 토큰 반복으로 무너집니다.

**되돌리기 ② step을 1500으로**

```python
max_steps = 1500   # 30000 대신. T4 30분 예산의 5%뿐
```

vocab을 줄였더라도 학습량이 절대적으로 부족하면 조건부 구조를 다 배우지 못합니다. 고정-$t$ top-1 정확도가 30000 step의 약 0.71에 한참 못 미치는 수준(8000 step에서도 약 0.26)에 머물러, 유니그램 marginal만 익힌 밋밋한 출력이 나옵니다.

이 collapse가 "조용한" 이유는 에러가 안 나기 때문입니다. 코드는 멀쩡히 돌고 loss도 줄어드는 듯 보이지만, 모델은 빈도표만 외운 상태입니다. loss 절대값을 유니그램 엔트로피와 비교하고, 고정-$t$ top-1 정확도와 생성 KL을 함께 봐야 이 조용한 실패를 잡아낼 수 있습니다.

## 다음 챕터 예고 — Ch 34

이번 장에서 작은 diffusion LM을 영어로 손수 살려냈다면, 다음 장에서는 같은 레시피를 **한국어**로 옮깁니다. Ch 34는 Ch 33과 같은 본체(`BertForMaskedLM`, hidden 256/4L)·같은 샘플러를 그대로 두고 학습 언어만 한국어(TinyStories-Korean)로 바꿉니다. 그런데 영어에서 100% `[MASK]`로도 버티던 레시피가 한국어에서는 무너집니다(train_loss 7.0대, 고정-$t$ acc 0.081). 한국어는 조사·어미를 빈도만으로 그럴듯하게 채울 수 있어 "주변을 안 봐도 되는" 지름길이 열리기 때문입니다. 그래서 Ch 34의 단 한 가지 변화는 **BERT가 원래 쓰던 80/10/10 마스킹**(80% `[MASK]` / 10% 랜덤 토큰 / 10% 원본 유지)으로, 이 지름길을 막아 train_loss 4.13 · acc 0.652로 생성을 회복시킵니다. 같은 코퍼스(Ch 26 한국어 AR과 동일)에서 *AR은 되는데 diffusion은 왜 무너졌나*를 나란히 대조하며 Phase 5를 마무리합니다.